# Milestone 3 lab, notebook 2 of 8: Six rows, wrong everywhere

## Where we are

| Notebook | What you do |
|---|---|
| 1. A price has a start and an end | products. See one forgotten date count every sale twice |
| **2. Six rows, wrong everywhere** (you are here) | stores. Find out why the weakest check is the best one here |
| 3. A gap is not a zero | stock_counts. Look for a row that is not there |
| 4. The same person twice | members. Find one person behind two keys, and keep their data private |
| 5. A file you do not control | supplier_prices. A supplier renames a column, and nothing fails |
| 6. Same shape, new meaning | supplier_prices. Every number is valid, and 100 times too high |
| 7. A contract for the supplier | Write down what a file must look like, and check it at the door |
| 8. Cards side by side | Compare the seven cards. Then see what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import (
    keep, lab_copy, on_lab, plant, prepare_week, production, rebuild_sales_lines,
    rebuilt_against_real, show_card, sql, standard_checks,
)

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 8 clean tables.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: Six rows

**Aiko Tanaka** manages store `S06`, Lotus Fenwick Harbour. In five weeks her store sold $23,537.17.

In this notebook one row is deleted by mistake from a table of six rows. Her store's sales become zero. No error, and five green checks.

The table is `stores`.

In [2]:
sql("SELECT * FROM clean.stores ORDER BY store_id")

,store_id,store_name,town,opens_at,closes_at,manager_name,tills
0,S01,Lotus Ashby High Street,Ashby,08:00:00,22:00:00,Marta Lindqvist,2
1,S02,Lotus Ashby Market Square,Ashby,08:00:00,22:00:00,Dana Okafor,2
2,S03,Lotus Calder Bridge,Calder,08:00:00,21:00:00,Tomas Reyes,2
3,S04,Lotus Calder Station,Calder,07:00:00,22:00:00,Ines Marchetti,2
4,S05,Lotus Fenwick Green,Fenwick,08:00:00,20:00:00,Samir Haddad,2
5,S06,Lotus Fenwick Harbour,Fenwick,08:00:00,21:00:00,Aiko Tanaka,2


In [3]:
show_card("stores", 1)

**From the check card `checks/stores.md`, question 1. What does one row mean?**

One store, with its opening hours and its manager. Six rows. The key is `store_id`.

Six rows. It was typed once, by head office, and it is rarely changed. It is tempting to say: there is nothing to check.

Question 2 of the card says who is hurt.

In [4]:
show_card("stores", 2)

**From the check card `checks/stores.md`, question 2. Who uses this table, and what do they decide with it?**

| Who | What they decide | What goes wrong for them when the data is bad |
|---|---|---|
| Every sales line, through the join in the cleaning step | Nothing. It is joined to its store | A store that is missing: all its sales are dropped. A store that is there twice: all its sales are doubled |
| Store managers | Their name is on the bonus report | The bonus goes to the wrong person |
| The opening-hours check on `sales_lines` | Whether a sale was made while the store was closed | Wrong hours make right sales look wrong |

Six rows, and a critical table. **Wrong here is wrong everywhere.**

Every sales line is joined to this table. So a wrong row here is wrong in every report. In milestone 1 you ranked it as a critical table, with six rows.

## Step 2: The weakest check becomes the best one

In milestone 1 the first standard check was "the table is not empty". On `sales_lines` it was almost useless: it passed while a whole store sent no file. You could not write "the table has N rows", because nobody knows how many lines Lotus will sell tomorrow.

Here somebody does know. Lotus has six stores. Head office says so. **For a table this small, the exact row count is a fact.**

Delete Aiko's store in a private copy, and rebuild the pipeline's join.

In [5]:
lab_copy("clean.stores")
plant("DELETE FROM lab.stores WHERE store_id = 'S06'")
rebuild_sales_lines()

lab.stores is a fresh copy of clean.stores: 6 rows.
Planted in the lab copy.
The real clean.sales_lines has 24,384 rows.
Rebuilt with your lab copy: 20,801 rows. That is -3,583.


In [6]:
rebuilt_against_real()

,store_id,real,rebuilt,difference
0,S01,"$33,427.35","$33,427.35",$0.00
1,S02,"$30,143.22","$30,143.22",$0.00
2,S03,"$26,855.52","$26,855.52",$0.00
3,S04,"$26,229.55","$26,229.55",$0.00
4,S05,"$21,784.12","$21,784.12",$0.00
5,S06,"$23,537.17",$0.00,"-$23,537.17"


3,583 lines are gone: every sale of `S06` in five weeks. The join found no store for them, so it dropped them.

What do the five standard checks say?

In [7]:
standard_checks(on_the_lab=True)

1. The table is not empty           0 bad rows  PASS
2. The key is not null              0 bad rows  PASS
3. The key is unique                0 bad rows  PASS
4. The status is an allowed value   0 bad rows  PASS
5. The total is positive            0 bad rows  PASS


All green. The table is not empty, and the five stores that are left look perfect.

Now the check. It is one line.

In [8]:
SIX_STORES = "SELECT count(*) AS stores_found, 6 AS stores_expected FROM clean.stores HAVING count(*) <> 6"
sql(on_lab(SIX_STORES))

,stores_found,stores_expected
0,5,6


Five found, six expected.

**The same check is useless on one table and exact on another.** That is why every table gets its own card.

## Step 3: Six rows again, and still wrong

A count alone is not enough. Suppose someone repairs the table in a hurry: they see five rows, and paste a row back. The wrong one.

In [9]:
plant("INSERT INTO lab.stores SELECT * FROM lab.stores WHERE store_id = 'S01'")
print(len(sql(on_lab(SIX_STORES))), "bad rows from the count check")
sql("SELECT store_id, count(*) AS times FROM lab.stores GROUP BY store_id ORDER BY store_id")

Planted in the lab copy.
0 bad rows from the count check


,store_id,times
0,S01,2
1,S02,1
2,S03,1
3,S04,1
4,S05,1


Six rows. The count check is happy. `S06` is still missing, and `S01` is there twice.

Rebuild the join.

In [10]:
rebuild_sales_lines()
rebuilt_against_real()

The real clean.sales_lines has 24,384 rows.
Rebuilt with your lab copy: 25,921 rows. That is +1,537.


,store_id,real,rebuilt,difference
0,S01,"$33,427.35","$66,854.70","$33,427.35"
1,S02,"$30,143.22","$30,143.22",$0.00
2,S03,"$26,855.52","$26,855.52",$0.00
3,S04,"$26,229.55","$26,229.55",$0.00
4,S05,"$21,784.12","$21,784.12",$0.00
5,S06,"$23,537.17",$0.00,"-$23,537.17"


**Marta Lindqvist**'s store `S01` is doubled: $33,427.35 too much. Aiko's store is still zero.

So the check needs both halves: exactly six rows, **and** no store twice.

In [11]:
SIX_STORES_ONCE = """
    SELECT 'the stores table' AS thing, count(*) AS stores_found, 6 AS stores_expected
    FROM clean.stores
    HAVING count(*) <> 6
    UNION ALL
    SELECT store_id AS thing, count(*) AS stores_found, 1 AS stores_expected
    FROM clean.stores
    GROUP BY store_id
    HAVING count(*) > 1
"""
sql(on_lab(SIX_STORES_ONCE))

,thing,stores_found,stores_expected
0,S01,2,1


On the real table it must be silent. The production check adds one column, the day, so that every finding can be said the same way.

In [12]:
yours = sql(SIX_STORES_ONCE)
keep("six_stores_once", yours)

Your query:           0 bad rows
The production check: 0 bad rows  (There are six stores, each once)
The same rows.


## Step 4: A file from a store nobody knows

The opposite direction. The table is fine, and a file arrives from a store that is not in it. Suppose a new store, `S07`, opens and its tills start sending totals before head office types the row.

Three kinds of file carry a `store_id`: sales lines, till totals and stock counts. Collect the ids they use, and join them to `stores`.

In [13]:
lab_copy("clean.stores", "raw.till_totals")
plant("UPDATE lab.raw_till_totals SET store_id = 'S07' WHERE store_id = 'S04' AND business_date = '2026-08-05'")

EVERY_STORE_IS_KNOWN = """
    SELECT used.store_id AS thing, used.found_in
    FROM (
        SELECT DISTINCT store_id, 'sales_lines' AS found_in FROM raw.sales_lines
        UNION
        SELECT DISTINCT store_id, 'till_totals' FROM raw.till_totals
        UNION
        SELECT DISTINCT store_id, 'stock_counts' FROM raw.stock_counts
    ) used
    LEFT JOIN clean.stores s ON s.store_id = used.store_id
    WHERE s.store_id IS NULL
"""
sql(on_lab(EVERY_STORE_IS_KNOWN))

lab.stores is a fresh copy of clean.stores: 6 rows.
lab.raw_till_totals is a fresh copy of raw.till_totals: 418 rows.
Planted in the lab copy.


,thing,found_in
0,S07,till_totals


You met this idea in milestone 2 as **referential integrity**: every row that points to another table finds a row there. There it was a sale pointing at a product. Here it is a file pointing at a store.

It reads the **raw** tables. In the clean tables the rows of an unknown store are already gone.

On the real tables it is silent.

In [14]:
keep("every_store_is_known", sql(EVERY_STORE_IS_KNOWN))

Your query:           0 bad rows
The production check: 0 bad rows  (Every file is from a store we know)
The same rows.


## Step 5: A table that should stand still

Question 3 of the card: how is the data made?

In [15]:
show_card("stores", 3)

**From the check card `checks/stores.md`, question 3. How is the data made? So what can go wrong?**

It was typed once, by head office, and it is rarely changed.

| Step | Made by | What can go wrong |
|---|---|---|
| A store is typed, once | A person | A wrong hour, a wrong name |
| A store is changed | A person, perhaps once a year | Nobody tells the data team. A row is deleted or added by mistake |
| The whole table is sent every night | A program | Almost nothing |

"Typed once, rarely changed." That sentence is a check.

A sales table is new every night. This table should be the same every night. So when it changes, that is either news or a mistake, and a person should hear about it.

The pipeline keeps every nightly file of `stores` in the raw table. Count them.

In [16]:
sql("SELECT count(DISTINCT _file_name) AS files, count(*) AS rows FROM raw.stores")

,files,rows
0,35,210


35 files of six rows. Suppose that in tonight's file, someone changed the closing time of `S02` from 22:00 to 23:00.

In [17]:
lab_copy("raw.stores", "clean.stores")
plant("""
    UPDATE lab.raw_stores SET closes_at = '23:00'
    WHERE store_id = 'S02' AND _file_name = (SELECT max(_file_name) FROM lab.raw_stores)
""")
plant("UPDATE lab.stores SET closes_at = TIME '23:00' WHERE store_id = 'S02'")
print(len(production("store_values_allowed", on_the_lab=True)), "bad rows from the check 'a store row makes sense'")

lab.raw_stores is a fresh copy of raw.stores: 210 rows.
lab.stores is a fresh copy of clean.stores: 6 rows.
Planted in the lab copy.
Planted in the lab copy.
0 bad rows from the check 'a store row makes sense'


The "makes sense" check is silent. 23:00 is a perfectly allowed hour.

Does it matter? Yes. Your check from milestone 2, "nothing is sold while the store is closed", reads these hours. With 23:00 in the table, a sale stamped 22:30 passes. One wrong cell here switches off a check somewhere else.

Take the newest file and the file before it, and keep the rows that are in the first and not in the second. SQL has a word for that: `EXCEPT`.

In [18]:
STORES_DID_NOT_CHANGE = """
    SELECT store_id AS thing, store_name, opens_at, closes_at, manager_name, tills
    FROM (
        SELECT store_id, store_name, town, opens_at, closes_at, manager_name, tills FROM raw.stores
        WHERE _file_name = (SELECT max(_file_name) FROM raw.stores)
        EXCEPT
        SELECT store_id, store_name, town, opens_at, closes_at, manager_name, tills FROM raw.stores
        WHERE _file_name = (
            SELECT max(_file_name) FROM raw.stores
            WHERE _file_name < (SELECT max(_file_name) FROM raw.stores)
        )
    ) changed
"""
sql(on_lab(STORES_DID_NOT_CHANGE))

,thing,store_name,opens_at,closes_at,manager_name,tills
0,S02,Lotus Ashby Market Square,08:00,23:00,Dana Okafor,2


In [19]:
keep("stores_did_not_change", sql(STORES_DID_NOT_CHANGE))

Your query:           0 bad rows
The production check: 0 bad rows  (The stores table is the same as yesterday)
The same rows.


This check would be nonsense on `sales_lines`, where every row is new every night. It fits a table that is typed once.

It cannot say the change is wrong. Perhaps the store really closes later now. It says: something changed, ask a person. On the card its seriousness is "low".

## Step 6: The whole card

In [20]:
show_card("stores", 6)

**From the check card `checks/stores.md`, question 6. The check**

Four checks. The SQL is in `lotus_pipeline/other_checks.py`.

| # | Check | The limit | Lens | Question | How serious |
|---|---|---|---|---|---|
| 22 | There are six stores, each once | Exactly 6 rows, and no store_id twice | Data | Unique? | High |
| 23 | A store row makes sense | A name, a town and a manager. It opens before it closes. It has at least one till | Data | Valid? | High |
| 24 | Every file is from a store we know | Every store_id in the three store files is in the stores table | Data | Consistent? | High |
| 25 | The stores table is the same as yesterday | No row is new, gone or different. A real change is confirmed by a person | Operational | Consistent? | Low. It is a reason to ask a person |

## What you now know

- `stores` has six rows and is a critical table. Wrong here is wrong everywhere.
- For a small table that people keep by hand, the exact row count is a fact, and the best check there is.
- A count is not enough alone: six rows can be five stores and one store twice.
- A table that is "typed once, rarely changed" gets a check no sales table could have: it is the same as yesterday.

**Say it in an interview:** "The stores table had six rows, and every sale joined to it. A row count told me nothing on the sales table, but here it was exact: six, each once. In a test, deleting one row made a store's $23,537 of sales vanish with five green checks. I also compared each night's file with the one before, because that table should stand still."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "A starter kit for any table".

Next: [notebook 3](03_a_gap_is_not_a_zero.ipynb). So far you looked for rows that are wrong. Next you look for a row that is not there.